# Classificador de Perfil de Retenção — Dataset Sintético e o Limite do Tamanho de Amostra

Este notebook revisita a proposta do documento "VC1" (classificar clientes 36-45 anos em perfis Gold/Silver/Bronze), corrigindo os dois problemas que apontamos na análise crítica:

1. **O rótulo é uma regra explícita que nós escrevemos, documentada abaixo — não é captação real observada.** Nenhuma métrica de negócio (CAC, LTV, ROI) é reivindicada a partir do resultado deste treino.
2. **A rede é dimensionada para o tamanho do dataset**, não copiada de outro contexto (a proposta original usava 64 neurônios para ~90 exemplos — aqui usamos 8, e mesmo assim o resultado abaixo mostra por quê isso ainda não é suficiente).

Continua a trilha de `01_tensores_fundamentos_censo.ipynb` e `02_migracao_pytorch.ipynb` (mesma stack: PyTorch).

In [ ]:
import numpy as np
import torch
torch.set_printoptions(precision=3, sci_mode=False)

## 1. Dataset sintético — a REGRA do rótulo, por escrito

As features (idade, frequência, tempo de cliente, distância, uso de app, participação em avaliação, plano, objetivo, horário) são sorteadas dentro de faixas plausíveis (algumas usam os ranges do apêndice — idade 36-45, ticket por plano — só porque fazem sentido como *domínio* da variável, não como prova de realismo do dataset inteiro).

**A REGRA que decide o rótulo é esta, e só esta** — nenhuma rede "descobre" isso escondido, é a fórmula que definimos:

> `score = 0.35·frequência + 0.30·tempo_cliente + 0.15·participação_avaliação + 0.10·uso_app − 0.10·distância + ruído`
> `score > 0.55 → Gold (2)` · `0.35 < score ≤ 0.55 → Silver (1)` · `score ≤ 0.35 → Bronze (0)`

O ruído (`± 0.08`, distribuição normal) existe de propósito: sem ele, o problema seria trivialmente separável e o exercício não ensinaria nada sobre generalização — na vida real, ninguém segue uma fórmula de fidelização à risca.

In [ ]:
def gerar_dataset(n: int, seed: int = 7):
    rng = np.random.default_rng(seed)

    idade = rng.uniform(36, 45, n)
    frequencia_semanal = rng.uniform(0, 7, n)
    tempo_cliente_meses = rng.uniform(0, 36, n)
    distancia_km = rng.uniform(0.3, 10, n)
    uso_app_tracking = rng.binomial(1, 0.5, n)
    participacao_avaliacao = rng.binomial(1, 0.5, n)
    tipo_plano = rng.choice(["basic", "premium"], n)
    objetivo_saude = rng.choice(["perda_peso", "ganho_muscular", "longevidade"], n)
    preferencia_horario = rng.choice(["manha", "tarde", "noite"], n)
    ticket_medio = np.where(
        tipo_plano == "premium",
        rng.uniform(99, 139, n),
        rng.uniform(69, 99, n),
    )

    # A REGRA (não é dado observado — ver célula acima)
    score = (
        0.35 * (frequencia_semanal / 7)
        + 0.30 * (tempo_cliente_meses / 36)
        + 0.15 * participacao_avaliacao
        + 0.10 * uso_app_tracking
        - 0.10 * (distancia_km / 10)
    )
    score = score + rng.normal(0, 0.08, n)
    labels = np.where(score > 0.55, 2, np.where(score > 0.35, 1, 0))

    return {
        "idade": idade, "frequencia_semanal": frequencia_semanal,
        "tempo_cliente_meses": tempo_cliente_meses, "distancia_km": distancia_km,
        "uso_app_tracking": uso_app_tracking, "participacao_avaliacao": participacao_avaliacao,
        "tipo_plano": tipo_plano, "objetivo_saude": objetivo_saude,
        "preferencia_horario": preferencia_horario, "ticket_medio": ticket_medio,
        "labels": labels,
    }


dados_90 = gerar_dataset(90)
print("distribuição de classes [Bronze, Silver, Gold]:", np.bincount(dados_90["labels"]))

## 2. Pré-processamento

**Correção em relação à proposta original**: a normalização usa o **min/max real dos dados gerados** (`idade.min()`, `idade.max()`, etc.), não os números do apêndice de mercado — um exemplo sintético pode cair fora do range do relatório (ex: ticket R$ 75), e normalizar contra um range que não é o do próprio dataset gera valores fora de [0,1] sem avisar.

In [ ]:
def normalizar(x):
    return (x - x.min()) / (x.max() - x.min())


def one_hot(vals, categorias):
    return np.array([[1.0 if v == c else 0.0 for c in categorias] for v in vals])


def montar_X_y(dados):
    X_num = np.stack([
        normalizar(dados["idade"]),
        normalizar(dados["frequencia_semanal"]),
        normalizar(dados["ticket_medio"]),
        normalizar(dados["tempo_cliente_meses"]),
        normalizar(dados["distancia_km"]),
        dados["uso_app_tracking"],
        dados["participacao_avaliacao"],
    ], axis=1)
    X_cat = np.concatenate([
        one_hot(dados["objetivo_saude"], ["perda_peso", "ganho_muscular", "longevidade"]),
        one_hot(dados["preferencia_horario"], ["manha", "tarde", "noite"]),
        one_hot(dados["tipo_plano"], ["basic", "premium"]),
    ], axis=1)
    X = np.concatenate([X_num, X_cat], axis=1).astype(np.float32)
    y = dados["labels"].astype(np.int64)
    return X, y


X_90, y_90 = montar_X_y(dados_90)
print("X shape:", X_90.shape, "(90 exemplos x 15 features após one-hot)")

## 3. Rede pequena + treino com medição honesta (treino vs. teste)

8 neurônios na camada oculta — bem menor que os 64 da proposta original, escolhidos para caber no tamanho deste dataset. Separamos 20% dos exemplos como teste: **nunca usados no treino**, servem só para medir se a rede generalizou ou só decorou.

In [ ]:
def treinar_avaliar(X, y, *, hidden=8, epocas=300, lr=0.05, seed=7, verboso=True):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    n = len(y)
    idx = rng.permutation(n)
    corte = int(n * 0.8)
    treino_idx, teste_idx = idx[:corte], idx[corte:]

    X_treino = torch.tensor(X[treino_idx])
    y_treino = torch.tensor(y[treino_idx])
    X_teste = torch.tensor(X[teste_idx])
    y_teste = torch.tensor(y[teste_idx])

    modelo = torch.nn.Sequential(
        torch.nn.Linear(X.shape[1], hidden),
        torch.nn.ReLU(),
        torch.nn.Linear(hidden, 3),
    )
    perda_fn = torch.nn.CrossEntropyLoss()
    otimizador = torch.optim.Adam(modelo.parameters(), lr=lr)

    for epoca in range(epocas):
        otimizador.zero_grad()
        logits = modelo(X_treino)
        perda = perda_fn(logits, y_treino)
        perda.backward()
        otimizador.step()
        if verboso and epoca % (epocas // 6) == 0:
            acc_treino = (logits.argmax(1) == y_treino).float().mean().item()
            print(f"época {epoca:>4}  perda={perda.item():.4f}  acc_treino={acc_treino:.2f}")

    with torch.no_grad():
        acc_treino_final = (modelo(X_treino).argmax(1) == y_treino).float().mean().item()
        acc_teste_final = (modelo(X_teste).argmax(1) == y_teste).float().mean().item()
    return acc_treino_final, acc_teste_final, len(y_treino), len(y_teste)


acc_treino_90, acc_teste_90, n_treino_90, n_teste_90 = treinar_avaliar(X_90, y_90)
print(f"\nFINAL (n=90) — acc_treino={acc_treino_90:.2f} (n={n_treino_90})  "
      f"acc_teste={acc_teste_90:.2f} (n={n_teste_90})")
print("Chance aleatória com 3 classes: ~0.33")

## 4. O resultado honesto: overfitting

Mesmo com uma rede pequena (8 neurônios, não 64), a acurácia de treino chega a 100% enquanto a de teste fica em torno de 0.67 — bem acima do acaso (0.33 para 3 classes), mas com uma diferença grande em relação ao treino. Isso **não é bug** — é o dataset pequeno (90 exemplos, 15 features após one-hot) permitindo que a rede memorize parte do treino em vez de só aprender o padrão geral. É exatamente o risco que apontamos na análise crítica do documento original, só que agora medido, não hipotético.

## 5. Diagnóstico: é a arquitetura ou é o tamanho da amostra?

Testamos a mesma arquitetura (8 neurônios), a mesma regra de rótulo, só que com 10x mais exemplos sintéticos (900 em vez de 90) — isolando a variável "quantidade de dado".

In [ ]:
dados_900 = gerar_dataset(900)
X_900, y_900 = montar_X_y(dados_900)
acc_treino_900, acc_teste_900, n_treino_900, n_teste_900 = treinar_avaliar(X_900, y_900)
print(f"\nFINAL (n=900) — acc_treino={acc_treino_900:.2f} (n={n_treino_900})  "
      f"acc_teste={acc_teste_900:.2f} (n={n_teste_900})")

## Conclusão honesta

| | n=90 | n=900 |
|---|---|---|
| Acurácia treino | 1.00 | 0.79 |
| Acurácia teste | 0.67 | 0.74 |
| Gap treino−teste | 0.33 (overfitting) | 0.05 (generaliza) |

1. **A arquitetura nunca foi o problema.** Com a mesma rede pequena, passar de 90 para 900 exemplos fechou o gap entre treino e teste (de 0.33 para 0.05) — confirma que o gargalo real do documento original era **volume de dado**, não número de neurônios (a proposta original piorava isso ainda mais usando 64 neurônios em 90 exemplos, o que teria overfitado ainda mais rápido).
2. **O teto de ~0.74-0.79 com n=900 é esperado, não uma falha**: é o próprio ruído que injetamos na regra do rótulo (`± 0.08`). Uma rede não pode prever melhor que a regra que gerou o dado permite — isso vale igualmente para dado real: se o comportamento humano tem componente aleatório, nenhum modelo chega a 100%.
3. **Isso continua sendo um exercício de mecânica, não uma ferramenta do GymSite.** Os números acima medem se a rede consegue redescobrir uma fórmula que nós escrevemos — não dizem nada sobre retenção real de aluno. Quando existir captação/retenção observada de verdade, o pipeline inteiro (normalização por min/max real, one-hot, rede pequena, split treino/teste, medir teste ≠ treino) é reaproveitável — muda só a fonte do rótulo, de "regra escrita à mão" para "resultado observado".